In [2]:
import re
import os
import numpy as np
import pandas as pd
import shutil
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.neighbors import KDTree
from tqdm import tqdm

In [3]:
# === Getting standards info ===

def get_standards(stand_path):

    #standards_path = '/home/frz603/DSP/matrix_standards/standards_positive.xlsx'
    standards_table = pd.read_excel(stand_path, skiprows=4, header = 0)

    # Reformating the info
    indices = [1, 2, 3, 6] + list(range(31, 41))
    standards = standards_table.iloc[:, indices].reset_index(drop = True)
    standards = standards[standards['Metabolite name']!='Unknown'].reset_index(drop = True)
    standards["Average Rt(min)"]=standards["Average Rt(min)"]*60
    standards["Metabolite name"] = standards["Metabolite name"].str.replace("w/o MS2:", "", regex=False).reset_index(drop=True)

    return standards


In [4]:

standards_path = '/home/frz603/DSP/matrix_standards/standards_positive.xlsx'
standards = get_standards(standards_path)
standards


,Average Rt(min),Average Mz,Metabolite name,Fill %,MS/MS spectrum,230503_Batch2_100,230503_Batch2_85,230503_Batch2_86,230503_Batch2_94,230503_Batch2_95,230503_Batch2_96,230503_Batch2_97,230503_Batch2_98,230503_Batch2_99
0,873.00,117.00352,Succinic acid,0.333,NaN,0,0,0,4916,4349,3031,0,0,0
1,966.78,117.00381,Succinic acid,0.333,NaN,0,0,0,3987,3276,2106,0,0,0
2,157.68,119.03705,Succinic acid,0.222,NaN,0,0,0,3027,2345,0,0,0,0
3,324.54,119.03815,Succinic acid,0.556,NaN,0,0,0,35034,28061,16718,8528,4771,0
4,704.64,119.05167,Succinic acid,0.444,NaN,0,0,0,13690,11208,6956,3257,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,1174.68,633.16052,Rutin,0.111,NaN,31094,0,0,0,0,0,0,0,0
177,822.06,649.12244,RUTOSIDE (rutin),0.333,NaN,0,0,0,11812,8725,4264,0,0,0
178,715.32,649.12939,RUTOSIDE (rutin),0.667,NaN,0,0,0,478694,400218,242155,101850,34028,4032
179,1175.40,649.13556,RUTOSIDE (rutin),0.111,NaN,9232,0,0,0,0,0,0,0,0


In [6]:
# Selecting only flavonoids from standards (Ewa says they are the most prominent)
flavonoids = ['Luteolin', 'Kaempferol', 'Kaempferol-7-neohesperidoside', 'Naringenin', 'Luteolin', 'Luteolin-6-C-glucoside', 'Rutin', 'ellagic acid', 'ellagic_acid', '(-)-Epigallocatechin-3-O-gallate', '(-)-epigallocatechin gallate', 'quercitrin', 'Quercitrin', 'Quercetin 3-O-alpha-rhamnopyranoside', 'Naringenin', 'Apigenin']

In [7]:
# Selecting one for now
stand_name = flavonoids[0]
stand_mz = np.median(np.array(standards.loc[standards['Metabolite name'] == stand_name, "Average Mz"]))
stand_rt = np.array(standards.loc[standards['Metabolite name'] == stand_name, 'Average Rt(min)'])

stand = standards[standards['Metabolite name'] == stand_name].sort_values(by = 'Average Rt(min)')
stand

,Average Rt(min),Average Mz,Metabolite name,Fill %,MS/MS spectrum,230503_Batch2_100,230503_Batch2_85,230503_Batch2_86,230503_Batch2_94,230503_Batch2_95,230503_Batch2_96,230503_Batch2_97,230503_Batch2_98,230503_Batch2_99
132,340.08,287.06134,Luteolin,0.444,NaN,0,0,0,37821,33024,17780,4350,0,0
129,558.90,287.06104,Luteolin,0.556,NaN,0,0,0,32564,24854,13499,5682,3218,0
131,682.80,287.06125,Luteolin,0.444,NaN,0,0,0,20978,14608,7300,3073,0,0
128,769.62,287.06094,Luteolin,0.333,NaN,0,0,0,9940,7229,3778,0,0,0
133,822.54,287.06186,Luteolin,0.667,NaN,0,0,0,747508,569939,298282,120448,56074,10510
134,1025.88,287.06226,Luteolin,0.778,NaN,7994,0,0,3971776,3192775,1854988,845740,415890,82678


In [8]:
path_ms_files = "/home/frz603/DSP/matrix_standards/raw_images/clustering/labels_20/sn3_w6/single_labels_overlap"

# === List all PNG files ===
all_files = [f for f in os.listdir(path_ms_files) if f.endswith(".npy")]

# === Separate into MS1 and MS2 lists ===
ms1_files = [f for f in all_files if "MS1_" in f]
ms2_files = [f for f in all_files if "MS2_" in f]

# === Print summary ===
print(f"Found {len(ms1_files)} MS1 files")
print(f"Found {len(ms2_files)} MS2 files")

Found 2833 MS1 files
Found 1447 MS2 files


In [9]:
# === Parameters ===
BINARY = False  # If True, convert to grayscale and binarize before sum

def image_projection_embedding(img_path):
    #img = Image.open(img_path).convert("L")  # grayscale
    arr = np.load(img_path)

    if BINARY:
        arr = (arr > 128).astype(int)

    row_sum = arr.sum(axis=1)  # shape (H,)
    col_sum = arr.sum(axis=0)  # shape (W,)
    vec = np.concatenate([row_sum, col_sum])

    # Normalize so max is 1
    max_val = vec.max()
    if max_val > 0:
        vec = vec / max_val

    return vec

def extract_embeddings(file_list, folder):
    embeddings = []
    valid_names = []
    for fname in tqdm(file_list):
        try:
            path = os.path.join(folder, fname)
            vec = image_projection_embedding(path)
            embeddings.append(vec)
            valid_names.append(fname)
        except Exception as e:
            print(f"Error processing {fname}: {e}")
    return np.array(embeddings), valid_names

In [10]:
ms1_vectors, ms1_names = extract_embeddings(ms1_files, path_ms_files)
ms2_vectors, ms2_names = extract_embeddings(ms2_files, path_ms_files)

ms1_names

  0%|          | 0/2833 [00:00<?, ?it/s]

100%|██████████| 1447/1447 [00:04<00:00, 294.48it/s]


['standardsnpy_MS1_130_0_130_1_004.npy',
 'standardsnpy_MS1_689_1_689_2_000.npy',
 'standardsnpy_MS1_257_2_257_3_000.npy',
 'standardsnpy_MS1_429_0_429_1_001.npy',
 'standardsnpy_MS1_634_0_634_1_001.npy',
 'standardsnpy_MS1_134_9_135_0_004.npy',
 'standardsnpy_MS1_487_0_487_1_002.npy',
 'standardsnpy_MS1_181_1_181_2_000.npy',
 'standardsnpy_MS1_130_0_130_1_003.npy',
 'standardsnpy_MS1_612_0_612_1_001.npy',
 'standardsnpy_MS1_191_0_191_1_000.npy',
 'standardsnpy_MS1_758_1_758_2_000.npy',
 'standardsnpy_MS1_379_1_379_2_001.npy',
 'standardsnpy_MS1_287_3_287_4_001.npy',
 'standardsnpy_MS1_279_0_279_1_036.npy',
 'standardsnpy_MS1_213_0_213_1_002.npy',
 'standardsnpy_MS1_201_9_202_0_000.npy',
 'standardsnpy_MS1_303_1_303_2_003.npy',
 'standardsnpy_MS1_485_2_485_3_000.npy',
 'standardsnpy_MS1_627_0_627_1_002.npy',
 'standardsnpy_MS1_384_2_384_3_000.npy',
 'standardsnpy_MS1_132_0_132_1_003.npy',
 'standardsnpy_MS1_729_2_729_3_000.npy',
 'standardsnpy_MS1_165_0_165_1_000.npy',
 'standardsnpy_M

In [11]:
# === Setting parameters ===
K = 5
output_folder = f'/home/frz603/DSP/matrix_standards/raw_images/clustering/kdtree/sn3_w6_k{K}'

# === Nearest neighbor search ===
tree = KDTree(ms1_vectors)
dists, indices = tree.query(ms2_vectors, k=K, sort_results=True)

matches = pd.DataFrame([
    (ms2_names[i], ms1_names[indices[i][j]], dists[i][j])
    for i in range(len(ms2_names))
    for j in range(K)
], columns=["ms2_name", "ms1_name", "distance"])

matches.to_excel(os.path.join(output_folder, 'final_matches.xlsx'))
matches

,ms2_name,ms1_name,distance
0,standardsnpy_MS2_549_2_549_3_000.npy,standardsnpy_MS1_890_3_890_4_000.npy,0.883451
1,standardsnpy_MS2_549_2_549_3_000.npy,standardsnpy_MS1_892_3_892_4_000.npy,0.884383
2,standardsnpy_MS2_549_2_549_3_000.npy,standardsnpy_MS1_891_3_891_4_000.npy,0.885402
3,standardsnpy_MS2_549_2_549_3_000.npy,standardsnpy_MS1_906_2_906_3_000.npy,0.891328
4,standardsnpy_MS2_549_2_549_3_000.npy,standardsnpy_MS1_907_2_907_3_001.npy,0.902144
...,...,...,...
7230,standardsnpy_MS2_204_0_204_1_005.npy,standardsnpy_MS1_247_0_247_1_006.npy,0.367357
7231,standardsnpy_MS2_204_0_204_1_005.npy,standardsnpy_MS1_324_1_324_2_003.npy,0.426026
7232,standardsnpy_MS2_204_0_204_1_005.npy,standardsnpy_MS1_292_1_292_2_004.npy,0.426360
7233,standardsnpy_MS2_204_0_204_1_005.npy,standardsnpy_MS1_273_1_273_2_005.npy,0.453190


In [28]:
# === Selecting all MS1 labels in the mass slice that matches the standards ===
image_name = np.unique([f for f in ms1_names if float(f.split('_')[2]+'.'+f.split('_')[3])<=stand_mz and float(f.split('_')[4]+'.'+re.sub('.npy', '',f.split('_')[5]))>=stand_mz])
print(image_name)

['standardsnpy_MS1_287_0_287_1_000.npy'
 'standardsnpy_MS1_287_0_287_1_001.npy'
 'standardsnpy_MS1_287_0_287_1_002.npy'
 'standardsnpy_MS1_287_0_287_1_003.npy'
 'standardsnpy_MS1_287_0_287_1_004.npy'
 'standardsnpy_MS1_287_0_287_1_005.npy'
 'standardsnpy_MS1_287_0_287_1_006.npy'
 'standardsnpy_MS1_287_0_287_1_007.npy']


In [29]:
# === Getting all MS2 labels that match all selected MS1 labels ===
#spectra = matches[matches['ms1_name'].str.removesuffix(".npy").str.replace("standardsnpy_", "", regex=False).isin(image_name)].iloc[:, 1:4].reset_index(drop = True)
spectra = matches[matches['ms1_name'].isin(image_name)].sort_values(by="ms2_name").reset_index(drop = True)
spectra

,ms2_name,ms1_name,distance
0,standardsnpy_MS2_121_0_121_1_000.npy,standardsnpy_MS1_287_0_287_1_001.npy,0.078806
1,standardsnpy_MS2_121_0_121_1_002.npy,standardsnpy_MS1_287_0_287_1_004.npy,0.090316
2,standardsnpy_MS2_121_0_121_1_004.npy,standardsnpy_MS1_287_0_287_1_007.npy,0.172330
3,standardsnpy_MS2_123_0_123_1_001.npy,standardsnpy_MS1_287_0_287_1_001.npy,0.089319
4,standardsnpy_MS2_127_0_127_1_005.npy,standardsnpy_MS1_287_0_287_1_007.npy,0.200730
...,...,...,...
77,standardsnpy_MS2_665_9_666_0_000.npy,standardsnpy_MS1_287_0_287_1_007.npy,0.278197
78,standardsnpy_MS2_682_9_683_0_000.npy,standardsnpy_MS1_287_0_287_1_007.npy,0.154812
79,standardsnpy_MS2_683_9_684_0_000.npy,standardsnpy_MS1_287_0_287_1_007.npy,0.141993
80,standardsnpy_MS2_684_9_685_0_000.npy,standardsnpy_MS1_287_0_287_1_007.npy,0.174479


In [31]:
# === Getting info on RT of every single peak to be able to match them with MsDial ===
rt_range = [100,1100]
rt_bin_size = 0.73
rt_bins = np.arange(rt_range[0], rt_range[1]+rt_bin_size, rt_bin_size) 

sample_no = 1 # taking QC for the int for m/z (maybe chosen differently?)

rt_av = []

for i in range(len(image_name)):
    img = np.load(os.path.join(f'/home/frz603/DSP/matrix_standards/raw_images/clustering/labels_20/sn3_w6/single_labels_overlap/{image_name[i]}'))
    bin_range = [min(np.where(img>0)[1])+max(np.where(img>0)[1])]
    rt_av = np.append(rt_av, rt_bins[int(np.ceil(np.sum(bin_range)/2))])

rt_info = pd.DataFrame({'image_name': image_name, 'av_rt': rt_av})
rt_info


,image_name,av_rt
0,standardsnpy_MS1_287_0_287_1_000.npy,342.36
1,standardsnpy_MS1_287_0_287_1_001.npy,561.36
2,standardsnpy_MS1_287_0_287_1_002.npy,684.00
3,standardsnpy_MS1_287_0_287_1_003.npy,765.03
4,standardsnpy_MS1_287_0_287_1_004.npy,827.08
5,standardsnpy_MS1_287_0_287_1_005.npy,857.74
6,standardsnpy_MS1_287_0_287_1_006.npy,1029.29
7,standardsnpy_MS1_287_0_287_1_007.npy,1054.84


In [34]:
ms2_image = []
int_spectra = []

for i in range(len(rt_info)):
    int_image = []
    ms2_chosen = spectra[spectra['ms1_name']==rt_info['image_name'].iloc[i]].reset_index(drop = True)
    for k in range(len(ms2_chosen)):
        img = np.load(f'/home/frz603/DSP/matrix_standards/raw_images/clustering/labels_20/sn3_w6/single_labels_overlap/{ms2_chosen.loc[k,'ms2_name']}')[sample_no]
        int_image.append(np.max(img))
    ms2_chosen = np.unique([int(f.split('_')[2]) for f in ms2_chosen['ms2_name']])
    ms2_image.append(ms2_chosen)
    int_spectra.append(int_image)

rt_info['mz'] = ms2_image
rt_info['int'] = int_spectra

rt_info

,image_name,av_rt,mz,int
0,standardsnpy_MS1_287_0_287_1_000.npy,342.36,[],[]
1,standardsnpy_MS1_287_0_287_1_001.npy,561.36,"[121, 123, 135, 149, 161, 245, 271, 272, 284, ...","[127.2631378173828, 125.72613525390624, 169.92..."
2,standardsnpy_MS1_287_0_287_1_002.npy,684.00,[287],[108.341796875]
3,standardsnpy_MS1_287_0_287_1_003.npy,765.03,[],[]
4,standardsnpy_MS1_287_0_287_1_004.npy,827.08,"[121, 213, 287, 288, 325, 331, 399]","[138.2740936279297, 115.26991271972656, 110.0,..."
5,standardsnpy_MS1_287_0_287_1_005.npy,857.74,"[153, 271]","[216.60421752929688, 101.76797485351562]"
6,standardsnpy_MS1_287_0_287_1_006.npy,1029.29,"[89, 135, 161, 213, 241, 259, 269, 287, 288, 2...","[332.87713623046875, 471.6954956054688, 403.52..."
7,standardsnpy_MS1_287_0_287_1_007.npy,1054.84,"[121, 127, 128, 137, 147, 157, 158, 160, 165, ...","[521.4660034179688, 414.7892456054688, 647.134..."


In [37]:
# === MS2 fragments from 2DLC ===
MS2_2d = pd.read_excel('/home/frz603/DSP/matrix_standards/2dlc.xlsx')
MS2_fragments = sorted(np.array([float(i.strip()) for i in MS2_2d.loc[MS2_2d['Name'] == stand_name, 'Fragments (m/z)'].iloc[0].split(',')]))

MS2_fragments

[np.float64(285.0391),
 np.float64(286.0427),
 np.float64(299.0186),
 np.float64(301.0357),
 np.float64(302.0392)]

In [39]:
stand_rt

array([ 769.62,  558.9 ,  682.8 ,  340.08,  822.54, 1025.88])

In [44]:
# === Choosing the most 'reliable' peak from MsDial and  ===
peak_ind = np.where(np.abs(np.diff(rt_info['av_rt'], stand_rt[stand['Fill %']==max(stand['Fill %'])][0])))

TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [55]:

np.where(np.abs(stand_rt[stand['Fill %']==max(stand['Fill %'])][0]-rt_info['av_rt'])==np.max(np.abs(stand_rt[stand['Fill %']==max(stand['Fill %'])][0]-rt_info['av_rt'])))

(array([0]),)

In [52]:
rt_info['av_rt']

0     342.36
1     561.36
2     684.00
3     765.03
4     827.08
5     857.74
6    1029.29
7    1054.84
Name: av_rt, dtype: float64